In [2]:
# ============================================================
# Re-clone repository into Colab runtime
# ============================================================

%cd /content

!rm -rf capstone-quantum

!git clone https://github.com/manuelarceaguirre/capstone-quantum.git

%cd /content/capstone-quantum

!ls

/content
Cloning into 'capstone-quantum'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 70 (delta 25), reused 56 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 2.32 MiB | 7.01 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/capstone-quantum
data  docs  environment.yml  figures  notebooks  README.md  Week7.md


In [4]:
# ============================================================
# Cell 1: Imports, repository setup, and QSK path loading
# Signature Kernel Baseline Prototype
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch

# ------------------------------------------------------------
# Display settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Repository paths
# ------------------------------------------------------------

REPO_ROOT = Path("/content/capstone-quantum")
DATA_DIR = REPO_ROOT / "data" / "processed"

TRACK_B_PATH = DATA_DIR / "track_B_curated.parquet"
QSK_PATHS_PATH = DATA_DIR / "qsk_paths_track_B.npz"

# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------

print("=" * 60)
print("PATH VALIDATION")
print("=" * 60)

print("TRACK_B_PATH exists:", TRACK_B_PATH.exists())
print("QSK_PATHS_PATH exists:", QSK_PATHS_PATH.exists())

# ------------------------------------------------------------
# Load Track B dataframe
# ------------------------------------------------------------

track_b_df = pd.read_parquet(TRACK_B_PATH)

track_b_df.index = pd.to_datetime(track_b_df.index)

print("\n" + "=" * 60)
print("TRACK B DATA")
print("=" * 60)

print("Shape:", track_b_df.shape)

print("\nDate range:")
print(track_b_df.index.min(), "->", track_b_df.index.max())

print("\nColumns:")
print(track_b_df.columns.tolist())

display(track_b_df.head())

# ------------------------------------------------------------
# Load QSK path tensor
# ------------------------------------------------------------

qsk_data = np.load(QSK_PATHS_PATH, allow_pickle=True)

paths = qsk_data["paths"]
path_index = qsk_data["index"]
path_columns = qsk_data["columns"]

print("\n" + "=" * 60)
print("QSK PATH TENSOR")
print("=" * 60)

print("paths shape:", paths.shape)
print("paths dtype:", paths.dtype)

print("\nindex shape:", path_index.shape)
print("index dtype:", path_index.dtype)

print("\nfirst 5 index values:")
print(path_index[:5])

print("\nlast 5 index values:")
print(path_index[-5:])

print("\ncolumns shape:", path_columns.shape)
print("columns dtype:", path_columns.dtype)

print("\ncolumns:")
print(path_columns)

PATH VALIDATION
TRACK_B_PATH exists: True
QSK_PATHS_PATH exists: True

TRACK B DATA
Shape: (767, 6)

Date range:
1962-04-01 00:00:00 -> 2026-02-01 00:00:00

Columns:
['UNRATE', 'PERMIT', 'S&P 500', 'UMCSENTx', 'T10Y3M_level', 'T10Y3M_delta']


,UNRATE,PERMIT,S&P 500,UMCSENTx,T10Y3M_level,T10Y3M_delta
date,,,,,,
1962-04-01,0.0,7.118826,-0.032387,0.0,1.11,NaN
1962-05-01,-0.1,7.040536,-0.077267,-4.5,1.18,0.07
1962-06-01,0.0,7.050989,-0.124253,0.0,1.18,0.00
1962-07-01,-0.1,7.080868,0.023802,0.0,1.09,-0.09
1962-08-01,0.3,7.090077,0.026844,-3.8,1.16,0.07



QSK PATH TENSOR
paths shape: (743, 24, 7)
paths dtype: float64

index shape: (743,)
index dtype: object

first 5 index values:
['1964-04-01' '1964-05-01' '1964-06-01' '1964-07-01' '1964-08-01']

last 5 index values:
['2025-10-01' '2025-11-01' '2025-12-01' '2026-01-01' '2026-02-01']

columns shape: (7,)
columns dtype: <U12

columns:
['_time' 'UNRATE' 'PERMIT' 'S&P 500' 'UMCSENTx' 'T10Y3M_level'
 'T10Y3M_delta']


In [5]:
# ============================================================
# Cell 2: Align QSK paths with UNRATE target
# ============================================================

TARGET_COL = "UNRATE"

# Convert path index to DatetimeIndex
path_dates = pd.to_datetime(path_index)

# Build target series from Track B
target_series = track_b_df[TARGET_COL].copy()
target_series.index = pd.to_datetime(target_series.index)

# Align target values to path end dates
y = target_series.loc[path_dates].values

# Basic validation
assert len(paths) == len(y), "Path tensor and target vector lengths do not match."

print("=" * 60)
print("PATH/TARGET ALIGNMENT")
print("=" * 60)

print("Target:", TARGET_COL)
print("X paths shape:", paths.shape)
print("y shape:", y.shape)

print("\nDate range:")
print(path_dates.min(), "->", path_dates.max())

print("\nFirst 5 aligned rows:")
alignment_preview = pd.DataFrame({
    "date": path_dates[:5],
    "target": y[:5],
})
display(alignment_preview)

print("\nMissing target values:", np.isnan(y).sum())

PATH/TARGET ALIGNMENT
Target: UNRATE
X paths shape: (743, 24, 7)
y shape: (743,)

Date range:
1964-04-01 00:00:00 -> 2026-02-01 00:00:00

First 5 aligned rows:


,date,target
0,1964-04-01,-0.1
1,1964-05-01,-0.2
2,1964-06-01,0.1
3,1964-07-01,-0.3
4,1964-08-01,0.1



Missing target values: 0


In [6]:
# ============================================================
# Cell 3: Benchmark-style pseudo-OOS splits
# ============================================================

def generate_poos_splits(
    n_samples,
    initial_train_size=120,
    test_size=1,
    step=1
):
    """
    Generate rolling pseudo-out-of-sample splits.
    """

    splits = []

    for train_start in range(
        0,
        n_samples - initial_train_size - test_size + 1,
        step
    ):

        train_end = train_start + initial_train_size

        test_start = train_end
        test_end = test_start + test_size

        split = {
            "train_idx": np.arange(train_start, train_end),
            "test_idx": np.arange(test_start, test_end),
        }

        splits.append(split)

    return splits


# ------------------------------------------------------------
# Generate splits
# ------------------------------------------------------------

poos_splits = generate_poos_splits(
    n_samples=len(paths),
    initial_train_size=120,
    test_size=1,
    step=1
)

print("=" * 60)
print("PSEUDO-OOS SPLITS")
print("=" * 60)

print("Number of splits:", len(poos_splits))

print("\nFirst split:")
print(poos_splits[0])

print("\nLast split:")
print(poos_splits[-1])

# ------------------------------------------------------------
# Verify date ranges
# ------------------------------------------------------------

first_split = poos_splits[0]

train_dates = path_dates[first_split["train_idx"]]
test_dates = path_dates[first_split["test_idx"]]

print("\nFirst split train dates:")
print(train_dates.min(), "->", train_dates.max())

print("\nFirst split test date:")
print(test_dates[0])

PSEUDO-OOS SPLITS
Number of splits: 623

First split:
{'train_idx': array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119]), 'test_idx': array([120])}

Last split:
{'train_idx': array([622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634,
       635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647,
       648, 649, 650, 651, 652, 653, 654, 655, 656, 657, 658, 6

In [7]:
# ============================================================
# Cell 4: First classical path-kernel baseline
# Flattened-path RBF Kernel Ridge prototype
# ============================================================

from sklearn.metrics.pairwise import rbf_kernel

ALPHA = 1.0
GAMMA = "scale"

predictions = []
actuals = []
forecast_dates = []

for split in poos_splits:
    train_idx = split["train_idx"]
    test_idx = split["test_idx"]

    X_train_paths = paths[train_idx]
    X_test_paths = paths[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Flatten path tensor for first kernel prototype:
    # (n_samples, window_length, path_dim) -> (n_samples, window_length * path_dim)
    X_train_flat = X_train_paths.reshape(X_train_paths.shape[0], -1)
    X_test_flat = X_test_paths.reshape(X_test_paths.shape[0], -1)

    # Scale using training data only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_test_scaled = scaler.transform(X_test_flat)

    # RBF Kernel Ridge Regression
    model = KernelRidge(
        alpha=ALPHA,
        kernel="rbf",
        gamma=None
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)[0]

    predictions.append(y_pred)
    actuals.append(y_test[0])
    forecast_dates.append(path_dates[test_idx][0])

results_df = pd.DataFrame({
    "date": forecast_dates,
    "actual": actuals,
    "prediction": predictions
})

results_df["abs_error"] = np.abs(results_df["actual"] - results_df["prediction"])
results_df["sq_error"] = (results_df["actual"] - results_df["prediction"]) ** 2

mae = results_df["abs_error"].mean()
rmse = np.sqrt(results_df["sq_error"].mean())

# Benchmark-consistent MASE denominator
y_full = track_b_df[TARGET_COL].values
naive_mae = np.mean(np.abs(y_full[1:] - y_full[:-1]))
mase = mae / naive_mae

print("=" * 60)
print("FLATTENED PATH RBF KERNEL RIDGE RESULTS")
print("=" * 60)

print(f"Forecast observations: {len(results_df)}")
print(f"MAE:        {mae:.6f}")
print(f"RMSE:       {rmse:.6f}")
print(f"Naive MAE:  {naive_mae:.6f}")
print(f"MASE:       {mase:.6f}")

display(results_df.head())
display(results_df.tail())

FLATTENED PATH RBF KERNEL RIDGE RESULTS
Forecast observations: 623
MAE:        0.143902
RMSE:       0.467154
Naive MAE:  0.208747
MASE:       0.689363


,date,actual,prediction,abs_error,sq_error
0,1974-04-01,0.0,0.009816,0.009816,0.000096
1,1974-05-01,0.0,0.030892,0.030892,0.000954
2,1974-06-01,0.3,0.050069,0.249931,0.062465
3,1974-07-01,0.1,0.047183,0.052817,0.002790
4,1974-08-01,0.0,0.027530,0.027530,0.000758


,date,actual,prediction,abs_error,sq_error
618,2025-10-01,0.0,0.069102,0.069102,0.004775
619,2025-11-01,0.1,-0.001503,0.101503,0.010303
620,2025-12-01,-0.1,-0.025156,0.074844,0.005602
621,2026-01-01,-0.1,-0.109196,0.009196,0.000085
622,2026-02-01,0.1,-0.097030,0.197030,0.038821
